In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [2]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
category_translations = pd.read_csv('../data/raw/product_category_name_translation.csv')
geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')

In [5]:
tables = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'payments': payments,
    'revviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category_translations': category_translations
}

In [6]:
summary = []

for table_name, df in tables.items():
    summary.append({
        'table_name': table_name,
        'rows': df.shape[0],
        'columns': df.shape[1],
        'duplicate_rows': df.duplicated().sum(),
        'memory_mb': round(df.memory_usage(deep=True).sum() / 1024 / 1024, 2),
    })

summary_df = pd.DataFrame(summary).sort_values(by='rows', ascending=False)

summary_df

,table_name,rows,columns,duplicate_rows,memory_mb
1,geolocation,1000163,5,261831,130.26
2,order_items,112650,7,0,35.99
3,payments,103886,5,0,16.23
0,customers,99441,5,0,26.59
5,orders,99441,8,0,52.94
4,revviews,99224,7,0,39.12
6,products,32951,9,0,6.30
7,sellers,3095,4,0,0.59
8,category_translations,71,2,0,0.01


In [10]:
for table_name, df in tables.items():
    print(f'\n===== {table_name.upper()} ======')
    print(df.columns.to_list())


===== CUSTOMERS ======
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

===== GEOLOCATION ======
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

===== ORDER_ITEMS ======
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

===== PAYMENTS ======
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

===== REVVIEWS ======
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

===== ORDERS ======
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

===== PRODUCTS ======
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght

In [9]:
for table_name, df in tables.items():
    print(f'\n===== {table_name.upper()} =====')
    df.info()


===== CUSTOMERS =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB

===== GEOLOCATION =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3  

In [11]:
null_summary = []

for table_name, df in tables.items():
    for column in df.columns:
        null_count = df[column].isna().sum()
        null_percentage = round((null_count / len(df)) * 100, 2)

        if null_count > 0:
            null_summary.append({
                'table_name': table_name,
                'column_name': column,
                'null_count': null_count,
                'null_percentage': null_percentage
            })

null_summary_df = pd.DataFrame(null_summary).sort_values(by=['table_name', 'null_count'], ascending=[True, False])

null_summary_df

,table_name,column_name,null_count,null_percentage
4,orders,order_delivered_customer_date,2965,2.98
3,orders,order_delivered_carrier_date,1783,1.79
2,orders,order_approved_at,160,0.16
5,products,product_category_name,610,1.85
6,products,product_name_lenght,610,1.85
7,products,product_description_lenght,610,1.85
8,products,product_photos_qty,610,1.85
9,products,product_weight_g,2,0.01
10,products,product_length_cm,2,0.01
11,products,product_height_cm,2,0.01


In [12]:
key_candidates = [
    ('customers', customers, 'customer_id'),
    ('orders', orders, 'order_id'),
    ('products', products, 'product_id'),
    ('sellers', sellers, 'seller_id'),
    ('category_translation', category_translations, 'product_category_name')
]

primary_key_check = []

for table_name, df, key in key_candidates:
    primary_key_check.append({
        'table_name': table_name,
        'key_column': key,
        'rows': len(df),
        'unique_values': df[key].nunique(),
        'null_values': df[key].isna().sum(),
        'is_valid_primary_key': len(df) == df[key].nunique() and df[key].isna().sum() == 0
    })

primary_key_check_df = pd.DataFrame(primary_key_check)

primary_key_check_df

,table_name,key_column,rows,unique_values,null_values,is_valid_primary_key
0,customers,customer_id,99441,99441,0,True
1,orders,order_id,99441,99441,0,True
2,products,product_id,32951,32951,0,True
3,sellers,seller_id,3095,3095,0,True
4,category_translation,product_category_name,71,71,0,True


In [13]:
order_items_key_duplicates = order_items[['order_id', 'order_item_id']].duplicated().sum()

payments_key_duplicates = payments[['order_id', 'payment_sequential']].duplicated().sum()

print('Duplicidades em order_items usando order_id + order_item_id:', order_items_key_duplicates)
print('Duplicidades em payments usando order_id + payment_sequential:', payments_key_duplicates)

Duplicidades em order_items usando order_id + order_item_id: 0
Duplicidades em payments usando order_id + payment_sequential: 0


In [14]:
print('Review_id duplicados:', reviews['review_id'].duplicated().sum())
print('Order_id duplicados:', reviews['order_id'].duplicated().sum())

Review_id duplicados: 814
Order_id duplicados: 551


In [15]:
def check_relationship(child_df, child_col, parent_df, parent_col, relationship_name):
    child_values = set(child_df[child_col].dropna().unique())
    parent_values = set(parent_df[parent_col].dropna().unique())
    
    missing_values = child_values - parent_values
    
    return {
        'relationship': relationship_name,
        'child_column': child_col,
        'parent_column': parent_col,
        'child_distinct_values': len(child_values),
        'parent_distinct_values': len(parent_values),
        'missing_values_in_parent': len(missing_values)
    }

In [16]:
relationships = []

relationships.append(
    check_relationship(
        orders, 'customer_id',
        customers, 'customer_id',
        'orders -> customers'
    )
)

relationships.append(
    check_relationship(
        order_items, 'order_id',
        orders, 'order_id',
        'order_items -> orders'
    )
)

relationships.append(
    check_relationship(
        order_items, 'product_id',
        products, 'product_id',
        'order_items -> products'
    )
)

relationships.append(
    check_relationship(
        order_items, 'seller_id',
        sellers, 'seller_id',
        'order_items -> sellers'
    )
)

relationships.append(
    check_relationship(
        payments, 'order_id',
        orders, 'order_id',
        'payments -> orders'
    )
)

relationships.append(
    check_relationship(
        reviews, 'order_id',
        orders, 'order_id',
        'reviews -> orders'
    )
)

relationships.append(
    check_relationship(
        products, 'product_category_name',
        category_translations, 'product_category_name',
        'products -> category_translation'
    )
)

relationships_df = pd.DataFrame(relationships)

relationships_df

,relationship,child_column,parent_column,child_distinct_values,parent_distinct_values,missing_values_in_parent
0,orders -> customers,customer_id,customer_id,99441,99441,0
1,order_items -> orders,order_id,order_id,98666,99441,0
2,order_items -> products,product_id,product_id,32951,32951,0
3,order_items -> sellers,seller_id,seller_id,3095,3095,0
4,payments -> orders,order_id,order_id,99440,99441,0
5,reviews -> orders,order_id,order_id,98673,99441,0
6,products -> category_translation,product_category_name,product_category_name,73,71,2


In [18]:
missing_categories = (
    set(products['product_category_name'].dropna().unique())
    - set(category_translations['product_category_name'].dropna().unique())
)

missing_categories

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}

In [19]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

for col in date_columns:
    print(col)
    print('Data mínima:', orders[col].min())
    print('Data máxima:', orders[col].max())
    print('Nulos:', orders[col].isna().sum())
    print('-' * 50)

order_purchase_timestamp
Data mínima: 2016-09-04 21:15:19
Data máxima: 2018-10-17 17:30:18
Nulos: 0
--------------------------------------------------
order_approved_at
Data mínima: 2016-09-15 12:16:38
Data máxima: 2018-09-03 17:40:06
Nulos: 160
--------------------------------------------------
order_delivered_carrier_date
Data mínima: 2016-10-08 10:34:01
Data máxima: 2018-09-11 19:48:28
Nulos: 1783
--------------------------------------------------
order_delivered_customer_date
Data mínima: 2016-10-11 13:46:32
Data máxima: 2018-10-17 13:22:46
Nulos: 2965
--------------------------------------------------
order_estimated_delivery_date
Data mínima: 2016-09-30 00:00:00
Data máxima: 2018-11-12 00:00:00
Nulos: 0
--------------------------------------------------


In [20]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [21]:
print('Linhas:', len(geolocation))
print('Prefixos de CEP únicos:', geolocation['geolocation_zip_code_prefix'].nunique())
print('Duplicatas:', geolocation.duplicated().sum())

Linhas: 1000163
Prefixos de CEP únicos: 19015
Duplicatas: 261831


In [22]:
## A base Olist possui 9 tabelas principais. 
## A tabela orders é o centro da análise de pedidos.
## A tabela order_items é o centro da análise de vendas.
## A tabela payments permite análise financeira dos pedidos.
## A tabela reviews permite análise de satisfação.
## As tabelas customers, products, sellers, geolocation e category_translation funcionam como dimensões.
## A tabela geolocation precisa ser tratada, pois tem muitas linhas duplicadas.
## A tabela reviews precisa de atenção porque review_id aparece duplicado.
## As datas precisam ser convertidas para datetime.
## Para análises comerciais, provavelmente a base principal deverá considerar pedidos delivered.